# 01 Enable incoming A2A on Strandvik's Customer Service agent

Maria's agent (Kolvo) calls Strandvik over **A2A** (agent to agent). The Foundry portal lets an agent *call* others, but switching on the **incoming** A2A endpoint is not in the portal yet. This notebook does that one step for `strandvik-customer-service`, which you created in the portal.

Steps:
1. **Business role:** write the agent card below (what other companies' agents see about your agent).
2. **Builder role:** run the cells. The notebook publishes the card, checks it, and prints the exact Azure portal steps to let Maria's agent in.

## 1. The agent card (Business role writes this)

In [ ]:
AGENT_NAME = "strandvik-customer-service"

# Who is this agent, for an outside agent? Keep it true to your instructions:
# who may ask, what it can do (price and delivery requests, open a case), what it never does.
CARD_DESCRIPTION = (
    "Strandvik Drives customer service. Receives requests from approved customers' agents, "
    "checks who is asking, opens a case and hands it to the account owner. Never quotes, orders or commits."
)

SKILLS = [
    {"id": "request-quote",
     "name": "Request price and delivery",
     "description": "An approved customer's agent asks for price and delivery of a drive unit. Opens a case."},
    # Add more skills if your agent does more, for example a case status check.
]

# Which agent may call it? (project, agent). Challenge 2: Kolvo's procurement agent.
# Challenge 6: Alex's agent in your own project, ("strandvik", "alex-agent").
CALLER = ("kolvo", "maria-procurement")


In [ ]:
# Run once if the packages are missing (the Codespace installs them for you).
# Packages come through the Microsoft package proxy.
# %pip install -q -r requirements.txt --index-url https://packagefeedproxy.microsoft.io/pypi/simple/

In [ ]:
# Settings come from ../.env, written by create-env.sh (or create-env.ps1).
# You sign in once with "az login"; the notebook reuses that sign-in.
import os, json, time
from pathlib import Path
import requests
from dotenv import load_dotenv
from azure.identity import AzureCliCredential, ChainedTokenCredential, DefaultAzureCredential

HACK_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(HACK_DIR / ".env", override=True)
env = os.environ.get

credential = ChainedTokenCredential(AzureCliCredential(process_timeout=30),
                                    DefaultAzureCredential(exclude_cli_credential=True))

def call(method, url, scope, body=None, ok=(200, 201, 202, 204), headers=None, data=None):
    """One REST call with an Entra token. Retries throttling. Raises on any other error."""
    hdrs = {"Authorization": "Bearer " + credential.get_token(scope).token}
    if body is not None:
        hdrs["Content-Type"] = "application/json"
    hdrs.update(headers or {})
    for attempt in range(8):
        r = requests.request(method, url, headers=hdrs, json=body, data=data, timeout=180)
        # 430 = the Fabric capacity is busy (small F2 capacity), 429/5xx = throttling.
        if r.status_code in (429, 430, 500, 502, 503) and attempt < 7:
            time.sleep(int(r.headers.get("Retry-After", 10 + 10 * attempt)))
            continue
        break
    if r.status_code not in ok:
        raise RuntimeError(f"{method} {url.split('?')[0]} -> HTTP {r.status_code}: {r.text[:1000]}")
    return r

In [ ]:
FOUNDRY_SCOPE = "https://ai.azure.com/.default"
STRANDVIK = env("PROJECT_ENDPOINT_STRANDVIK").rstrip("/")
KOLVO = env("PROJECT_ENDPOINT_KOLVO").rstrip("/")
API = "api-version=v1"

def foundry(method, url, body=None, ok=(200, 201)):
    return call(method, url, FOUNDRY_SCOPE, body, ok).json()

# The agent must exist first (you created it in the Foundry portal, project "strandvik").
agent = foundry("GET", f"{STRANDVIK}/agents/{AGENT_NAME}?{API}")
print("found", agent["name"], "latest version", agent["versions"]["latest"]["version"])

## 2. Publish the card and switch on the A2A protocol

In [ ]:
# Same call as the SDK's project.agents.update_details(...).
body = {
    "agent_card": {"description": CARD_DESCRIPTION, "version": "1.0", "skills": SKILLS},
    "agent_endpoint": {"protocol_configuration": {"responses": {}, "a2a": {}}},
}
foundry("PATCH", f"{STRANDVIK}/agents/{AGENT_NAME}?{API}", body)

A2A_URL = f"{STRANDVIK}/agents/{AGENT_NAME}/endpoint/protocols/a2a"
card = foundry("GET", f"{A2A_URL}/agentCard/v1.0?{API}")
print(json.dumps(card, indent=2)[:1500])
print("\nA2A base URL (write it down; the caller's A2A tool points here):\n", A2A_URL)

## 3. Who is calling? Let the caller's agent in

Challenge 2: the caller is Maria's agent (Kolvo). Challenge 6: it is Alex's agent in your own project (set `CALLER` in the first code cell).

Incoming A2A only accepts callers with an Azure role. Maria's agent has its own Entra identity (an *agent identity*).
Give that identity the **Foundry Agent Consumer** role: it can call agents, nothing else.

**Business role decides the scope:** only this agent (`agent`, least privilege) or every agent in your project (`project`).

Why a notebook and not the portal? The Foundry portal (agent > Details > Identity and access > Assign Azure roles) and the
Azure portal (Access control (IAM)) only find people and apps today, not agent identities. The REST call below is the same
role assignment the portals would make. Afterwards you can see it in the portal.


In [ ]:
SCOPE = "agent"   # "agent" = only the agent in AGENT_NAME (least privilege), "project" = every agent in the strandvik project

def principal_of(agent_json):
    latest = agent_json.get("versions", {}).get("latest", {})
    return (latest.get("instance_identity") or agent_json.get("instance_identity") or {}).get("principal_id")

ours = principal_of(foundry("GET", f"{STRANDVIK}/agents/{AGENT_NAME}?{API}"))
CALLER_PROJECT, CALLER_AGENT = CALLER
caller_endpoint = KOLVO if CALLER_PROJECT == "kolvo" else STRANDVIK
caller = principal_of(foundry("GET", f"{caller_endpoint}/agents/{CALLER_AGENT}?{API}"))
print(f"Your agent's identity ({AGENT_NAME}):", ours)
print(f"Caller's agent identity ({CALLER_AGENT}):", caller)


In [ ]:
import uuid

ARM = "https://management.azure.com"
ARM_SCOPE = "https://management.azure.com/.default"
CONSUMER_ROLE = "eed3b665-ab3a-47b6-8f48-c9382fb1dad6"   # Foundry Agent Consumer (built-in)

project_id = (f"/subscriptions/{env('AZURE_SUBSCRIPTION_ID')}/resourceGroups/{env('AZURE_RESOURCE_GROUP')}"
              f"/providers/Microsoft.CognitiveServices/accounts/{env('FOUNDRY_ACCOUNT')}/projects/strandvik")
scope_id = f"{project_id}/agents/{AGENT_NAME}" if SCOPE == "agent" else project_id

# Already granted? Then do nothing (safe to run twice).
existing = call("GET", f"{ARM}{scope_id}/providers/Microsoft.Authorization/roleAssignments"
                f"?api-version=2022-04-01&$filter=assignedTo('{caller}')", ARM_SCOPE).json()["value"]
if any(a["properties"]["roleDefinitionId"].endswith(CONSUMER_ROLE) for a in existing):
    print(CALLER_AGENT, "already has Foundry Agent Consumer on", SCOPE)
else:
    body = {"properties": {
        "roleDefinitionId": f"/subscriptions/{env('AZURE_SUBSCRIPTION_ID')}/providers/Microsoft.Authorization/roleDefinitions/{CONSUMER_ROLE}",
        "principalId": caller,
        "principalType": "ServicePrincipal",
        "description": f"{CALLER_AGENT} may call {AGENT_NAME} over A2A"}}
    call("PUT", f"{ARM}{scope_id}/providers/Microsoft.Authorization/roleAssignments/{uuid.uuid4()}?api-version=2022-04-01",
         ARM_SCOPE, body)
    print("Granted Foundry Agent Consumer to", CALLER_AGENT, "on", SCOPE, ":", scope_id.split("/providers/")[-1])

print(f"""
Now WAIT 3 to 6 minutes before testing. Until the role is active, {CALLER_AGENT}'s call fails with a vague
"tool_user_error". Test in the Foundry portal: project {CALLER_PROJECT} > Agents > {CALLER_AGENT} > Playground
(challenge 2: also the procurement board {env("WORKIQ_BOARD_URL")}).
""")
